# BlueStone Real Estate - Advanced Actual Modeling Notebook (4 Models)

This notebook implements the **4 models** defined in the ML architecture document:

1. **Property Price Prediction**
2. **Inquiry SLA Prediction**
3. **Property Recommendation Engine**
4. **Market Demand Forecast Model**

Dataset path used:
`C:\Users\user\Documents\LML BlueStoneKay\Data\`

**Compatibility fix included:** if your `scikit-learn` version does not support `squared=False`, RMSE is computed as:

```python
rmse = mean_squared_error(y_true, y_pred) ** 0.5
```


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib

warnings.filterwarnings('ignore')


## 1. Configure Path

In [ ]:
DATA_DIR = Path(r"C:\Users\user\Documents\LML BlueStoneKay\Data")
MODEL_DIR = DATA_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_DIR)
print(MODEL_DIR)


## 2. Load Datasets

In [ ]:
properties = pd.read_csv(DATA_DIR / 'properties_clean.csv')
prices = pd.read_csv(DATA_DIR / 'property_prices_history.csv')
users = pd.read_csv(DATA_DIR / 'users_clean.csv')
inquiries = pd.read_csv(DATA_DIR / 'customer_inquiries_clean.csv')
agents = pd.read_excel(DATA_DIR / 'agents.xlsx')

print('properties:', properties.shape)
print('prices:', prices.shape)
print('users:', users.shape)
print('inquiries:', inquiries.shape)
print('agents:', agents.shape)


## 3. Helper Functions

In [ ]:
def parse_budget_range(df, column='budget_range'):
    out = df.copy()
    parts = out[column].astype(str).str.extract(r'(?P<budget_min>\d+\.?\d*)-(?P<budget_max>\d+\.?\d*)')
    out['budget_min'] = pd.to_numeric(parts['budget_min'], errors='coerce')
    out['budget_max'] = pd.to_numeric(parts['budget_max'], errors='coerce')
    out['budget_mid'] = (out['budget_min'] + out['budget_max']) / 2
    out['budget_missing_flag'] = out['budget_min'].isna().astype(int)
    return out

def build_current_properties(df):
    out = df.copy()
    if 'is_current' in out.columns:
        out = out[out['is_current'] == True].copy()
    if 'data_quality_score' in out.columns:
        out = out.sort_values(['property_id', 'data_quality_score'], ascending=[True, False])
    return out.drop_duplicates('property_id', keep='first')

def build_latest_prices(df):
    out = df.copy()
    out['price_date'] = pd.to_datetime(out['price_date'], errors='coerce')
    for col in ['zestimate', 'assessed_value', 'list_price', 'price_per_sqft', 'price_change_pct']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    out = out.sort_values(['property_id', 'price_date'], ascending=[True, False])
    out = out.drop_duplicates('property_id', keep='first')
    return out.rename(columns={
        'price_date':'latest_price_date',
        'list_price':'latest_list_price',
        'zestimate':'latest_zestimate',
        'assessed_value':'latest_assessed_value',
        'price_per_sqft':'latest_price_per_sqft',
        'price_change_pct':'latest_price_change_pct'
    })

def build_agent_features(df, as_of_year=2026):
    out = df.copy()
    if 'hire_date' in out.columns:
        out['hire_date'] = pd.to_datetime(out['hire_date'], errors='coerce')
        out['agent_experience_years'] = as_of_year - out['hire_date'].dt.year
    return out

def make_preprocessor(X):
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ])
    return preprocessor


## 4. Canonical Curated Tables

In [ ]:
properties_current = build_current_properties(properties)
prices_latest = build_latest_prices(prices)
users_feat = parse_budget_range(users)
agents_feat = build_agent_features(agents)

print(properties_current.shape, prices_latest.shape, users_feat.shape, agents_feat.shape)


## Model 1 - Property Price Prediction

In [ ]:
price_df = properties_current.merge(prices_latest, on='property_id', how='inner')
price_df['property_age'] = 2026 - price_df['year_built']
price_df['bed_bath_ratio'] = price_df['bedrooms'] / np.maximum(price_df['bathrooms'], 1)
price_df['sqft_per_bedroom'] = price_df['sqft'] / np.maximum(price_df['bedrooms'], 1)
price_df['has_valid_geo'] = (
    price_df['latitude'].between(-90, 90) & price_df['longitude'].between(-180, 180)
).astype(int)

price_model_df = price_df[[
    'property_id','property_type','bedrooms','bathrooms','sqft','lot_size','year_built',
    'listing_status','latitude','longitude','data_quality_score','property_age',
    'bed_bath_ratio','sqft_per_bedroom','has_valid_geo','latest_list_price'
]].copy()

X = price_model_df.drop(columns=['latest_list_price', 'property_id'])
y = price_model_df['latest_list_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

price_model = Pipeline([
    ('preprocessor', make_preprocessor(X)),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=2))
])
price_model.fit(X_train, y_train)
price_pred = price_model.predict(X_test)

price_rmse = mean_squared_error(y_test, price_pred) ** 0.5
price_mae = mean_absolute_error(y_test, price_pred)
price_r2 = r2_score(y_test, price_pred)

print('Price Model RMSE:', round(price_rmse, 4))
print('Price Model MAE :', round(price_mae, 4))
print('Price Model R2  :', round(price_r2, 4))


## Model 2 - Inquiry SLA Prediction

In [ ]:
inq = inquiries.copy()
inq['inquiry_date'] = pd.to_datetime(inq['inquiry_date'], errors='coerce')
inq['response_date'] = pd.to_datetime(inq['response_date'], errors='coerce')
inq['inquiry_hour'] = inq['inquiry_date'].dt.hour
inq['inquiry_dayofweek'] = inq['inquiry_date'].dt.dayofweek
inq['has_property_id'] = inq['property_id'].notna().astype(int)
inq = inq[inq['response_time_hours'].notna()].copy()
inq['sla_breach_24h'] = (inq['response_time_hours'] > 24).astype(int)

sla_df = (
    inq.merge(users_feat[['user_id','user_type','budget_mid','budget_missing_flag']], on='user_id', how='left')
       .merge(properties_current[['property_id','property_type','bedrooms','bathrooms','sqft','listing_status','data_quality_score']], on='property_id', how='left')
       .merge(agents_feat[['agent_id','office_location','status','agent_experience_years']], on='agent_id', how='left', suffixes=('','_agent'))
)

sla_df['property_missing_flag'] = sla_df['property_type'].isna().astype(int)
sla_df = sla_df.rename(columns={'status_agent':'agent_status'})
sla_model_df = sla_df[[
    'inquiry_id','inquiry_type','status','inquiry_hour','inquiry_dayofweek','has_property_id',
    'user_type','budget_mid','budget_missing_flag','property_type','bedrooms','bathrooms',
    'sqft','listing_status','data_quality_score','office_location','agent_status',
    'agent_experience_years','property_missing_flag','sla_breach_24h'
]].copy()

X = sla_model_df.drop(columns=['sla_breach_24h', 'inquiry_id'])
y = sla_model_df['sla_breach_24h']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sla_model = Pipeline([
    ('preprocessor', make_preprocessor(X)),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, min_samples_leaf=2, class_weight='balanced'))
])
sla_model.fit(X_train, y_train)
sla_pred = sla_model.predict(X_test)

print('SLA Accuracy :', round(accuracy_score(y_test, sla_pred), 4))
print('SLA Precision:', round(precision_score(y_test, sla_pred, zero_division=0), 4))
print('SLA Recall   :', round(recall_score(y_test, sla_pred, zero_division=0), 4))
print('SLA F1 Score :', round(f1_score(y_test, sla_pred, zero_division=0), 4))
print(confusion_matrix(y_test, sla_pred))


## Model 3 - Property Recommendation Engine

In [ ]:
# Simple baseline recommendation engine using user preferences + current properties
reco_users = users_feat.copy()
reco_props = properties_current.copy()

if 'price' in reco_props.columns:
    reco_props['candidate_price'] = pd.to_numeric(reco_props['price'], errors='coerce')
else:
    reco_props = reco_props.merge(prices_latest[['property_id','latest_list_price']], on='property_id', how='left')
    reco_props['candidate_price'] = reco_props['latest_list_price']

def recommend_for_user(user_id, top_n=10):
    user_row = reco_users[reco_users['user_id'] == user_id].head(1)
    if user_row.empty:
        return pd.DataFrame()
    user_row = user_row.iloc[0]
    candidates = reco_props.copy()
    candidates['score'] = 0

    # budget fit
    if pd.notna(user_row.get('budget_min')) and pd.notna(user_row.get('budget_max')):
        candidates['score'] += ((candidates['candidate_price'] >= user_row['budget_min']) & (candidates['candidate_price'] <= user_row['budget_max'])).astype(int) * 3

    # location preference text match (simple baseline)
    pref_locs = str(user_row.get('preferred_locations', ''))
    if 'address_standardized' in candidates.columns:
        candidates['score'] += candidates['address_standardized'].astype(str).str.contains(pref_locs[:20], case=False, na=False).astype(int)

    # property type preference text match
    pref_prop = str(user_row.get('property_preferences', ''))
    if 'property_type' in candidates.columns:
        candidates['score'] += candidates['property_type'].astype(str).str.contains(pref_prop[:20], case=False, na=False).astype(int) * 2

    keep_cols = [c for c in ['property_id','property_type','bedrooms','bathrooms','sqft','candidate_price','score'] if c in candidates.columns]
    return candidates.sort_values('score', ascending=False)[keep_cols].head(top_n)

# sample recommendations (replace with a real user_id from your dataset if needed)
sample_user_id = reco_users['user_id'].iloc[0]
recommendations = recommend_for_user(sample_user_id, top_n=5)
print('Sample user_id:', sample_user_id)
recommendations


## Model 4 - Market Demand Forecast Model

In [ ]:
# Forecast future inquiry volume using monthly aggregated inquiry counts + lag features
demand = inquiries.copy()
demand['inquiry_date'] = pd.to_datetime(demand['inquiry_date'], errors='coerce')
demand = demand.dropna(subset=['inquiry_date']).copy()
demand['year_month'] = demand['inquiry_date'].dt.to_period('M').astype(str)

monthly_demand = demand.groupby('year_month').size().reset_index(name='inquiry_count')
monthly_demand['year_month'] = pd.to_datetime(monthly_demand['year_month'])
monthly_demand = monthly_demand.sort_values('year_month').reset_index(drop=True)
monthly_demand['month_num'] = monthly_demand['year_month'].dt.month
monthly_demand['quarter'] = monthly_demand['year_month'].dt.quarter
monthly_demand['lag_1'] = monthly_demand['inquiry_count'].shift(1)
monthly_demand['lag_2'] = monthly_demand['inquiry_count'].shift(2)
monthly_demand['lag_3'] = monthly_demand['inquiry_count'].shift(3)
monthly_demand = monthly_demand.dropna().copy()

X = monthly_demand[['month_num','quarter','lag_1','lag_2','lag_3']]
y = monthly_demand['inquiry_count']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
demand_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
demand_model.fit(X_train, y_train)
demand_pred = demand_model.predict(X_test)

demand_rmse = mean_squared_error(y_test, demand_pred) ** 0.5
demand_mae = mean_absolute_error(y_test, demand_pred)
demand_r2 = r2_score(y_test, demand_pred)

print('Demand Forecast RMSE:', round(demand_rmse, 4))
print('Demand Forecast MAE :', round(demand_mae, 4))
print('Demand Forecast R2  :', round(demand_r2, 4))
monthly_demand.tail()


## 10. Save Trained Models

In [ ]:
joblib.dump(price_model, MODEL_DIR / 'price_model.joblib')
joblib.dump(sla_model, MODEL_DIR / 'sla_model.joblib')
joblib.dump(demand_model, MODEL_DIR / 'demand_forecast_model.joblib')
print('Saved: price_model.joblib')
print('Saved: sla_model.joblib')
print('Saved: demand_forecast_model.joblib')


## Important Clarification

Your **previous notebook did not include all 4 models**.

It included only:
- Property Price Prediction
- Inquiry SLA Prediction

This upgraded notebook now includes the full **4-model architecture**:
- Property Price Prediction
- Inquiry SLA Prediction
- Property Recommendation Engine
- Market Demand Forecast Model
